单局测试

In [13]:
import chess
import pandas as pd

def extract_checks_and_mates(uid, moves_string):
    """
    解析单局游戏的着法，提取 Check 和 Checkmate 的步数。
    步数 (Ply) 从 1 开始计数。
    """
    # 1. 处理 None 或空值情况
    if pd.isna(moves_string) or not moves_string or not isinstance(moves_string, str):
        return None
        
    board = chess.Board()
    # 根据你的数据库格式，使用逗号分隔着法
    move_list = moves_string.strip().split(',')
    
    check_plies = []
    checkmate_plies = []
    
    # 2. 遍历着法，ply_count 从 1 开始
    for ply_count, move_str in enumerate(move_list, start=1):
        move_str = move_str.strip()
        if not move_str:
            continue
            
        try:
            # 你的数据是 UCI 格式 (例如 d2d4)，所以使用 from_uci
            move = chess.Move.from_uci(move_str)
            board.push(move)
            
            # 3. 状态判定 (注意：is_checkmate 成立时，is_check 也为 True)
            # 这里我做了互斥处理，如果是绝杀就只记入 mate 列表
            if board.is_checkmate():
                checkmate_plies.append(ply_count)
            elif board.is_check():
                check_plies.append(ply_count)
                
        except ValueError:
            # 如果遇到无法解析或非法的着法，立刻停止后续解析，防止棋盘状态崩溃
            break 
            
    # 4. 如果没有任何 check 或 checkmate，返回 None (即“不写了”)
    if not check_plies and not checkmate_plies:
        return None
        
    return {
        'uid': uid,
        'check_moves': check_plies,
        'checkmate_moves': checkmate_plies
    }

# --- 单局测试示例 ---
# 模拟你的数据库里的一行数据
sample_uid = 1
sample_moves = "e2e4,c7c5,g1f3,e7e6,c2c3,d7d5,e4d5,d8d5,d2d4,b8c6,c1e3,c5d4,c3d4,g8f6,b1c3,f8b4,f1d3,e8g8,e1g1,b4c3,b2c3,b7b6,c3c4,d5d8,e3g5,c8b7,a1b1,h7h6,g5h4,g8h8,d3c2,c6e7,f3e5,g7g5,h4g3,e7f5,d4d5,f5g3,f2g3,e6d5,d1d4,h8g8,e5c6,b7c6,f1f6,d5c4,d4d8,a8d8,f6c6,g8g7,c6c4,d8d2,c4c7,f8e8,b1f1,e8e2,c7f7,g7h8,f7f8,h8g7,f1f7"
move_list = [m.strip() for m in sample_moves.strip().split(',') if m.strip()]
total_plies = len(move_list)
print(f'总长度 {total_plies}')
result = extract_checks_and_mates(sample_uid, sample_moves)
print(f"测试结果: {result}")

# 模拟一个有空值的异常行
print(f"空值测试: {extract_checks_and_mates(2, None)}")

总长度 61
测试结果: {'uid': 1, 'check_moves': [57, 59], 'checkmate_moves': [61]}
空值测试: None


In [ ]:
import chess
import pandas as pd

def extract_checks_and_mates(uid, moves_string):
    """
    解析单局游戏的着法，提取 Check 和 Checkmate 的步数，并返回总步数。
    """
    if pd.isna(moves_string) or not moves_string or not isinstance(moves_string, str):
        return None
        
    board = chess.Board()
    # 拆分并过滤掉可能的空字符串（比如字符串末尾多了一个逗号）
    move_list = [m.strip() for m in moves_string.strip().split(',') if m.strip()]
    
    # 【新增】：计算总步数 (Ply)
    total_plies = len(move_list)
    
    check_plies = []
    checkmate_plies = []
    
    for ply_count, move_str in enumerate(move_list, start=1):
        try:
            move = chess.Move.from_uci(move_str)
            board.push(move)
            
            if board.is_checkmate():
                checkmate_plies.append(ply_count)
            elif board.is_check():
                check_plies.append(ply_count)
                
        except ValueError:
            # 如果遇到非法着法提前中断，我们可以把 total_plies 修正为实际走到的一步
            # 这样有助于发现坏数据
            total_plies = ply_count - 1 
            break 
            
    # 如果没有任何 check 或 checkmate，依然按你之前的要求不记录（返回 None）
    if not check_plies and not checkmate_plies:
        return None
        
    return {
        'uid': uid,
        'total_plies': total_plies,        # 【新增】：记录总步数
        'check_moves': check_plies,
        'checkmate_moves': checkmate_plies
    }

# --- 测试与验证逻辑 ---
sample_uid = 1
# 这是一个 26 步的例子（并没有 checkmate，只是为了测试长度）
sample_moves = "d2d4,g8f6,c2c4,e7e6,g1f3,b7b6,g2g3,c8b7,f1g2,f8b4,c1d2,b4e7,e1g1,e8g8,b1c3,c7c6,e2e4,d7d5,e4e5,f6e4,c4d5,c6d5,c3e4,d5e4,f3g5,e7g5"

# 这是一个四步杀 (Scholar's Mate) 的例子，转为 UCI 格式：e2e4,e7e5,d1h5,b8c6,f1c4,g8f6,h5f7
mate_moves = "e2e4,e7e5,d1h5,b8c6,f1c4,g8f6,h5f7" 

print("--- 测试 1: 普通对局 (没有 Checkmate) ---")
res1 = extract_checks_and_mates(1, sample_moves)
print(res1)
# 预期输出: {'uid': 1, 'total_plies': 26, 'check_moves': [10], 'checkmate_moves': []}

print("\n--- 测试 2: 绝杀对局 ---")
res2 = extract_checks_and_mates(2, mate_moves)
print(res2)
# 预期输出: {'uid': 2, 'total_plies': 7, 'check_moves': [], 'checkmate_moves': [7]}

# 检验“Checkmate 是否真的是最后一步”
if res2 and res2['checkmate_moves']:
    is_last_step = (res2['checkmate_moves'][0] == res2['total_plies'])
    print(f"Checkmate 发生在最后一步吗？ -> {is_last_step}")